# 🧊 TRELLIS 3D – باستخدام المستودع البديل (PRITHIVSAKTHIUR)
**قبل التشغيل:** تأكد من تفعيل GPU: **Runtime → Change runtime type → T4 GPU**.
سنستخدم المستودع الذي أشرت إليه إذا فشل المستودع الأصلي.

In [ ]:
# =============================================
# TRELLIS 3D – خلية واحدة (مستودع بديل)
# =============================================
import os, sys, subprocess, zipfile, io, requests, tempfile, traceback, importlib.metadata
import torch, numpy as np
from PIL import Image
import gradio as gr

# 1. فحص GPU
print("🔍 فحص GPU...")
if not torch.cuda.is_available():
    print("❌ لا يوجد GPU. الرجاء: Runtime → Change runtime type → GPU")
    exit(0)
!nvidia-smi
gpu_name = subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader")
print(f"✅ GPU: {gpu_name}")

SUPPORTED_GPU = ["A100", "A6000", "3090", "4090", "L4", "L40", "H100"]
USE_FLASH_ATTN = any(g in gpu_name for g in SUPPORTED_GPU)
print("✨ Flash Attention 2 مدعوم" if USE_FLASH_ATTN else "ℹ️  يستخدم SDPA.")

# 2. تثبيت الحزم الأساسية
def is_installed(pkg):
    try:
        importlib.metadata.version(pkg)
        return True
    except importlib.metadata.PackageNotFoundError:
        return False

print("\n🔍 فحص الحزم...")
base_pkgs = ["trimesh", "pygltflib", "viser", "omegaconf", "opencv-python", "imageio", "gradio", "huggingface_hub"]
for pkg in base_pkgs:
    if not is_installed(pkg):
        !pip install -q {pkg}
        print(f"✔️ {pkg}")
    else:
        print(f"✅ {pkg}")

# spconv
if not is_installed("spconv"):
    print("⏳ تثبيت spconv...")
    try:
        !pip install -q spconv
    except:
        !pip install -q spconv-cu118
    print("✔️ spconv")
else:
    print("✅ spconv")

# Flash Attention (اختياري)
if USE_FLASH_ATTN and not is_installed("flash-attn"):
    print("⚡ تثبيت Flash Attention 2...")
    !pip install -q ninja packaging
    !pip install flash-attn --no-build-isolation 2>&1 | tail -3
    import flash_attn
    print("✅ flash-attn")

# 3. استنساخ المستودع (الأصلي أولاً، ثم البديل)
REPOS = [
    {"url": "https://github.com/JeffreyXiang/TRELLIS.git", "dir": "/content/TRELLIS"},
    {"url": "https://github.com/PRITHIVSAKTHIUR/TRELLIS.2-Text-to-3D-FA2.git", "dir": "/content/TRELLIS.2"}
]

TARGET_DIR = None
for repo in REPOS:
    if os.path.isdir(repo["dir"]):
        print(f"✅ المستودع {repo['dir']} موجود مسبقًا.")
        TARGET_DIR = repo["dir"]
        break
    print(f"⬇️  محاولة استنساخ {repo['url']} ...")
    env = os.environ.copy()
    env["GIT_ASKPASS"] = "echo"
    env["GIT_TERMINAL_PROMPT"] = "0"
    result = subprocess.run(["git", "clone", "--depth", "1", repo["url"], repo["dir"]],
                            capture_output=True, text=True, env=env)
    if result.returncode == 0:
        print(f"✅ تم استنساخ {repo['dir']}")
        TARGET_DIR = repo["dir"]
        break
    else:
        print(f"❌ فشل: {result.stderr}")

if TARGET_DIR is None:
    raise RuntimeError("تعذر استنساخ أي مستودع. تحقق من الاتصال أو توفر المستودع.")

# 4. إعداد المسار وتثبيت الحزمة إذا لزم
os.chdir(TARGET_DIR)
sys.path.insert(0, TARGET_DIR)

# إذا كان هناك setup.py، نثبت الحزمة لضمان عمل import trellis
if os.path.isfile("setup.py"):
    print("📦 تثبيت الحزمة عبر setup.py...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], capture_output=True)
    # بعد التثبيت، يمكننا الاستيراد مباشرة دون تعقيد المسار
    print("✅ تم تثبيت الحزمة.")

# 5. استيراد trellis (مع البحث التلقائي عن المجلد)
def find_and_import_trellis():
    try:
        import trellis
        return True
    except ImportError:
        # البحث عن مجلد اسمه trellis داخل المشروع
        for root, dirs, files in os.walk(TARGET_DIR):
            if "trellis" in dirs:
                trellis_dir = os.path.join(root, "trellis")
                # يجب أن تحتوي على pipelines أو __init__.py
                if os.path.isdir(trellis_dir):
                    parent = os.path.dirname(trellis_dir)
                    if parent not in sys.path:
                        sys.path.insert(0, parent)
                    try:
                        import trellis
                        return True
                    except ImportError:
                        continue
        return False

if not find_and_import_trellis():
    raise ImportError("لم يتم العثور على حزمة trellis. قد يكون المستودع غير مكتمل.")
print("✅ استيراد trellis ناجح.")

# إعدادات الانتباه
if USE_FLASH_ATTN:
    try:
        import flash_attn
        attn_impl = "flash_attention_2"
        print("🔧 Flash Attention 2.")
    except:
        attn_impl = "sdpa"
        print("🔧 SDPA.")
else:
    attn_impl = "sdpa"
    print("🔧 SDPA.")

# 6. تحميل النموذج (قد يكون اسمه مختلفًا، سنحاول الاسم المعروف)
from trellis.pipelines import TrellisImageTo3DPipeline
model_name = "JeffreyXiang/TRELLIS-image-large"  # الافتراضي
try:
    pipeline = TrellisImageTo3DPipeline.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        attn_implementation=attn_impl
    ).to("cuda")
    print("✅ نموذج TRELLIS محمّل.")
except Exception as e:
    print(f"⚠️ خطأ في التحميل: {e}")
    # نجرب اسمًا آخر إن وجد
    try:
        model_name = "PRITHIVSAKTHIUR/TRELLIS.2-Text-to-3D-FA2"
        pipeline = TrellisImageTo3DPipeline.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            attn_implementation=attn_impl
        ).to("cuda")
        print("✅ نموذج بديل محمّل.")
    except:
        # محاولة أخيرة بدون تحديد attn_implementation
        pipeline = TrellisImageTo3DPipeline.from_pretrained(
            "JeffreyXiang/TRELLIS-image-large",
            torch_dtype=torch.float16
        ).to("cuda")
        print("✅ نموذج افتراضي محمّل.")

# 7. دالة التوليد
def generate_3d(image_input, seed, cfg_scale):
    if image_input is None:
        raise gr.Error("يرجى رفع صورة.")
    image = Image.fromarray(image_input).convert("RGB") if isinstance(image_input, np.ndarray) else image_input.convert("RGB")
    try:
        try:
            outputs = pipeline.run(image, seed=int(seed), cfg_scale=float(cfg_scale))
        except TypeError:
            outputs = pipeline.run(image, seed=int(seed))
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"فشل التوليد: {e}")
    mesh = outputs.get('mesh')
    if mesh is None:
        raise gr.Error("لم يتم توليد شبكة.")
    tmpdir = tempfile.mkdtemp()
    glb_path = os.path.join(tmpdir, "model.glb")
    mesh.export(glb_path)
    return glb_path, glb_path

# 8. واجهة Gradio
with gr.Blocks(title="TRELLIS 3D", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🧊 TRELLIS – تحويل الصورة إلى مجسم ثلاثي الأبعاد
    أرفع صورة واضغط **توليد**. يمكنك تدوير المجسم بالماوس.
    """)
    with gr.Row():
        with gr.Column():
            image_input = gr.Image(label="الصورة المدخلة", type="pil", image_mode="RGB")
            with gr.Accordion("⚙️ إعدادات", open=False):
                seed = gr.Number(value=42, label="Seed", precision=0)
                cfg_scale = gr.Slider(1.0, 20.0, 7.5, step=0.5, label="CFG Scale")
            generate_btn = gr.Button("🚀 توليد", variant="primary")
        with gr.Column():
            model_output = gr.Model3D(label="المجسم")
            download_btn = gr.File(label="تحميل GLB")
    generate_btn.click(fn=generate_3d, inputs=[image_input, seed, cfg_scale], outputs=[model_output, download_btn])

print("\n🌐 تشغيل واجهة Gradio...")
demo.queue(max_size=3).launch(share=True, server_port=7860, show_error=True)